# Esercizio RAG passo per passo
Quaderno guidato per indicizzare un PDF (ingestion) e interrogarlo (retrieval) usando la pipeline di `rag_pipeline.py`. Ogni cella spiega cosa accade prima di eseguirla.


## Prerequisiti
- Hai già installato le dipendenze con `uv sync` e attivato l'ambiente virtuale.
- Hai creato un file `.env` con `OPENAI_API_KEY`.
- Esegui il notebook dalla cartella `Notebook/`; la prima cella imposta automaticamente la working directory alla root del progetto per usare i percorsi definiti nella pipeline.


In [ ]:
# Preparazione: portiamo il notebook alla root del progetto e importiamo ciò che ci serve
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)  # i path di rag_pipeline.py sono relativi alla root
sys.path.append(PROJECT_ROOT)

from dotenv import load_dotenv
from datapizza.embedders.openai import OpenAIEmbedder
from datapizza.embedders import ChunkEmbedder
from datapizza.modules.parsers.docling import DoclingParser
from datapizza.modules.splitters import NodeSplitter
from datapizza.pipeline import IngestionPipeline
from datapizza.clients.openai import OpenAIClient

from rag_pipeline import (
    COLLECTION_NAME,
    QDRANT_PATH,
    EMBEDDING_MODEL,
    EMBEDDING_DIM,
    LLM_MODEL,
    get_vectorstore,
    setup_environment,
)

print(f'Working dir: {os.getcwd()}')
print(f'Qdrant path: {QDRANT_PATH}')


## Sezione 1: Ingestion
Obiettivo: prendere un PDF, trasformarlo in chunk vettoriali e salvarli in Qdrant con persistenza locale (`./qdrant_data`).

Passi che seguiremo:
1. Caricare le variabili d'ambiente e l'API key.
2. Dichiarare il percorso del PDF.
3. Creare embedder, parser, splitter e vector store.
   - Usiamo `get_vectorstore(create_collection=True)` che istanzia `QdrantVectorstore(location=None, path=QDRANT_PATH)`: Qdrant accetta un solo parametro tra `location/url/host/path`, quindi passiamo solo `path` per la persistenza locale.
4. Lanciare la pipeline di ingestion e salvare i vettori.


In [ ]:
# 1) Carica API key e verifica
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('Imposta OPENAI_API_KEY nel file .env prima di continuare')
print('API key trovata e caricata.')


In [ ]:
# 2) Percorso del PDF da indicizzare (modifica se usi un file diverso)
PDF_PATH = os.path.join('Datapizza/Ducati/data', 'Ducati_Overview.pdf')#MonsterRev02
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f'Non trovo il file: {PDF_PATH}. Metti il PDF in data/ oppure aggiorna il percorso.')
print(f'Userò questo file: {PDF_PATH}')


In [19]:
# 3) Costruiamo gli oggetti della pipeline e creiamo (o sovrascriviamo) la collection in locale
# get_vectorstore usa QdrantVectorstore(location=None, path=QDRANT_PATH) per rispettare il vincolo Qdrant di un solo parametro tra location/url/host/path
vectorstore = get_vectorstore(create_collection=True)
embedder = OpenAIEmbedder(api_key=api_key, model_name=EMBEDDING_MODEL)
pipeline = IngestionPipeline(
    modules=[
        DoclingParser(),  # estrae testo dal PDF
        NodeSplitter(max_char=1000),  # chunk di ~1000 caratteri
        ChunkEmbedder(client=embedder),  # embedding di ogni chunk
    ],
    vector_store=vectorstore,
    collection_name=COLLECTION_NAME,
)
print('Pipeline pronta: parser -> splitter -> embedder -> Qdrant')
print('Persistenza locale in:', QDRANT_PATH)


SyntaxError: invalid syntax (132392426.py, line 2)

In [ ]:
# 4) Eseguiamo l'ingestion
pipeline.run(PDF_PATH, metadata={'source': os.path.basename(PDF_PATH)})
print('Ingestion completata: i vettori sono salvati in', os.path.abspath(QDRANT_PATH))


### Cosa abbiamo fatto
- `DoclingParser` ha estratto il testo strutturato dal PDF.
- `NodeSplitter` ha creato chunk gestibili (max 1000 caratteri).
- `ChunkEmbedder` ha convertito ogni chunk in un vettore usando `text-embedding-3-small`.
- `QdrantVectorstore` ha salvato i vettori in `qdrant_data`, con collection chiamata `ducati_docs`.


## Sezione 2: Retrieval e risposta
Obiettivo: cercare i chunk più rilevanti per una domanda e generare una risposta con il modello LLM.

Passi che seguiremo:
1. Caricare il database locale e l'embedder per le query.
2. Calcolare l'embedding della domanda e cercare i `k` chunk più vicini.
3. Assemblare il contesto e chiamare l'LLM con un prompt vincolato alla fonte.


In [ ]:
# 1) Carichiamo vector store e modelli per la fase di retrieval
vectorstore = get_vectorstore(create_collection=False)
query_embedder = OpenAIEmbedder(api_key=api_key, model_name=EMBEDDING_MODEL)
llm = OpenAIClient(model=LLM_MODEL, api_key=api_key)
print('Vector store caricato e modelli pronti.')


In [ ]:
# 2) Scrivi qui la tua domanda
query = 'Quali sono i punti chiave del documento?'
query_vector = query_embedder.embed(query)
results = vectorstore.search(
    query_vector=query_vector,
    collection_name=COLLECTION_NAME,
    k=5,
)

print(f'Trovati {len(results)} chunk rilevanti. Mostro i primi due:')
for i, chunk in enumerate(results[:2], start=1):
    print(f'
--- Chunk {i} ---')
    print(chunk.text[:500] + ('...' if len(chunk.text) > 500 else ''))


In [ ]:
# 3) Costruiamo il contesto e interroghiamo l'LLM in modo vincolato alla fonte
context = '
---
'.join([chunk.text for chunk in results])
prompt = f'''Sei un assistente tecnico. Rispondi alla domanda dell'utente basandoti ESCLUSIVAMENTE sul contesto fornito.
Se l'informazione non è presente nel contesto, rispondi: "Non ho trovato questa informazione nel documento."
Non inventare informazioni.

CONTESTO:
{context}

DOMANDA: {query}

RISPOSTA:'''.strip()
response = llm.invoke(prompt)
print('Risposta:')
print(response.text)


### Cosa abbiamo fatto
- Calcolato l'embedding della query con lo stesso modello usato in ingestion.
- Recuperato i 5 chunk più simili da Qdrant.
- Concatenato i chunk in un unico contesto con separatori `---`.
- Inviato all'LLM un prompt che vieta l'hallucination e chiede di rispondere solo dal contesto.

Puoi ora modificare la domanda, cambiare `k` o provare altri PDF ripetendo la sezione di ingestion.
